Cookie Sales Performance Dashboard 

1.  Project Overview
This project focuses on building an interactive Jupyter notebook dashboard to analyze cookie sales data.
The dashboard provides insights into revenue, cost, customer behavior, and product performance using multiple datasets.

2. Project Objective
- To analyze sales performance of different cookie types  
- To track revenue, cost, and profit  
- To understand customer distribution  
- To build an interactive dashboard  
- To integrate Python visual  

3. Business Requirement
The business wants to:
- Identify top-performing cookie products
- Monitor sales trends over time
- Analyze customer distribution by location
- Track key metrics like revenue, cost, and profit
- Enable interactive filtering for better decision-making

4. Data Source
The project uses three datasets:
- Orders dataset (transaction-level data)
- Customers dataset (customer details)
- Cookie Types dataset (product-level information)

5. Data Preparation and Cleaning
- Removed inconsistencies using Trim and Clean functions
- Verified correct data types (Date, Numeric fields)
- Checked and handled missing values
- Ensured consistent naming across tables

6. Data Modelling
A star schema model was created:
- Orders table as the central fact table
- Customers and Cookie Types as dimension tables
Relationships:
- Customers[Customer ID] → Orders[Customer ID]
- Cookie Types[Cookie Type] → Orders[Product]
Both relationships are One-to-Many with single-direction filtering.

7. Key Measures
- Total Revenue = orders['Revenue'].sum()
- Total Cost = orders['Cost'].sum()
- Profit = Total Revenue – Total Cost
- Total Orders = orders['Order ID'].nunique()
- Avg Order Value = Total Revenue / Total Orders

8. Dashboard Development
The dashboard includes:
- KPI cards for Revenue, Cost, Orders, and Average Order Value
- Bar chart showing revenue by cookie type
- Line chart showing sales trends over time
- Donut chart showing order distribution by city
- Pie chart comparing revenue and cost
- Interactive filters for cookie type and city
  
Interactive filters:
- Cookie Type
- City

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

ModuleNotFoundError: No module named 'plotly'

In [ ]:
orders = pd.read_excel('Orders 1.xlsx')
customers = pd.read_excel('Customers 1.xlsx')
cookie_types = pd.read_excel('Cookie Types 1.xlsx')

orders.columns = orders.columns.str.strip()
customers.columns = customers.columns.str.strip()
cookie_types.columns = cookie_types.columns.str.strip()

orders['Order Date'] = pd.to_datetime(orders['Order Date'], errors='coerce')
orders['Revenue'] = pd.to_numeric(orders['Revenue'], errors='coerce').fillna(0)
orders['Cost'] = pd.to_numeric(orders['Cost'], errors='coerce').fillna(0)
orders['Quantity'] = pd.to_numeric(orders['Quantity'], errors='coerce').fillna(0)

orders = orders.merge(customers, on='Customer ID', how='left')
orders = orders.merge(cookie_types, left_on='Cookie Type', right_on='Cookie Type', how='left')

orders['Profit'] = orders['Revenue'] - orders['Cost']
orders['Order Month'] = orders['Order Date'].dt.to_period('M').dt.to_timestamp()

orders['City'] = orders['City'].fillna('Unknown')
orders['Cookie Type'] = orders['Cookie Type'].fillna('Unknown')

In [ ]:
def build_dashboard(cookie_types, cities):
    filtered = orders.copy()
    if cookie_types:
        filtered = filtered[filtered['Cookie Type'].isin(cookie_types)]
    if cities:
        filtered = filtered[filtered['City'].isin(cities)]

    total_revenue = filtered['Revenue'].sum()
    total_cost = filtered['Cost'].sum()
    total_profit = filtered['Profit'].sum()
    total_orders = filtered['Order ID'].nunique() if 'Order ID' in filtered.columns else len(filtered)
    avg_order_value = total_revenue / total_orders if total_orders else 0

    kpi = go.Figure()
    kpi.add_trace(go.Indicator(mode='number', title='Revenue', value=total_revenue, number={'prefix': '$'}))
    kpi.add_trace(go.Indicator(mode='number', title='Cost', value=total_cost, number={'prefix': '$'}))
    kpi.add_trace(go.Indicator(mode='number', title='Profit', value=total_profit, number={'prefix': '$'}))
    kpi.add_trace(go.Indicator(mode='number', title='Avg Order Value', value=avg_order_value, number={'prefix': '$'}))
    kpi.update_layout(grid={'rows': 1, 'columns': 4}, template='plotly_white', height=200)

    revenue_by_cookie = filtered.groupby('Cookie Type', as_index=False)['Revenue'].sum().sort_values('Revenue', ascending=False)
    fig_cookie = px.bar(revenue_by_cookie, x='Cookie Type', y='Revenue', title='Revenue by Cookie Type', text='Revenue')
    fig_cookie.update_layout(xaxis_title='Cookie Type', yaxis_title='Revenue', template='plotly_white')

    trends = filtered.groupby('Order Month', as_index=False)[['Revenue', 'Cost', 'Profit']].sum()
    fig_trend = px.line(trends, x='Order Month', y=['Revenue', 'Cost', 'Profit'], title='Monthly Revenue, Cost and Profit')
    fig_trend.update_layout(xaxis_title='Month', yaxis_title='Amount', template='plotly_white')

    city_dist = filtered.groupby('City', as_index=False)['Order ID'].nunique().rename(columns={'Order ID': 'Orders'})
    fig_city = px.pie(city_dist, names='City', values='Orders', title='Order Distribution by City')

    revenue_cost = filtered[['Revenue', 'Cost']].sum().reset_index()
    revenue_cost.columns = ['Metric', 'Amount']
    fig_revenue_cost = px.pie(revenue_cost, names='Metric', values='Amount', title='Revenue vs Cost')

    display(kpi)
    display(fig_cookie)
    display(fig_trend)
    display(fig_city)
    display(fig_revenue_cost)

    top_cookies = revenue_by_cookie.head(10)
    display(top_cookies.style.format({'Revenue': '${:,.0f}'}))

In [ ]:
cookie_options = sorted(orders['Cookie Type'].dropna().unique())
city_options = sorted(orders['City'].dropna().unique())

cookie_selector = widgets.SelectMultiple(
    options=cookie_options,
    value=cookie_options,
    description='Cookie Types',
    rows=6,
    layout=widgets.Layout(width='45%')
)
city_selector = widgets.SelectMultiple(
    options=city_options,
    value=city_options,
    description='Cities',
    rows=6,
    layout=widgets.Layout(width='45%')
)

controls = widgets.HBox([cookie_selector, city_selector])
dashboard_output = widgets.interactive_output(build_dashboard, {'cookie_types': cookie_selector, 'cities': city_selector})

display(widgets.Markdown('### Filters'))
display(controls)
display(dashboard_output)

9. Key Insights
- Chocolate Chip is the highest revenue-generating product
- Sales show fluctuations over time indicating demand variation
- A few cities contribute significantly to total orders
- Overall profit margin is strong
- Some cookie types have lower performance and can be improved

# Cookies Sales Interactive Dashboard

This notebook builds an interactive dashboard for cookie sales using the provided sales, customer, and product data. Use the filters to explore revenue, cost, profit, and order trends by cookie type and city.

## Dataset Overview
- `Orders 1.xlsx`: transaction-level orders data
- `Customers 1.xlsx`: customer details by customer ID
- `Cookie Types 1.xlsx`: product information for cookie types

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

In [ ]:
orders = pd.read_excel('Orders 1.xlsx')
customers = pd.read_excel('Customers 1.xlsx')
cookie_types = pd.read_excel('Cookie Types 1.xlsx')

orders.columns = orders.columns.str.strip()
customers.columns = customers.columns.str.strip()
cookie_types.columns = cookie_types.columns.str.strip()

orders['Order Date'] = pd.to_datetime(orders['Order Date'], errors='coerce')
orders['Revenue'] = pd.to_numeric(orders['Revenue'], errors='coerce').fillna(0)
orders['Cost'] = pd.to_numeric(orders['Cost'], errors='coerce').fillna(0)
orders['Quantity'] = pd.to_numeric(orders['Quantity'], errors='coerce').fillna(0)

orders = orders.merge(customers, on='Customer ID', how='left')
orders = orders.merge(cookie_types, left_on='Cookie Type', right_on='Cookie Type', how='left')

orders['Profit'] = orders['Revenue'] - orders['Cost']
orders['Order Month'] = orders['Order Date'].dt.to_period('M').dt.to_timestamp()

orders['City'] = orders['City'].fillna('Unknown')
orders['Cookie Type'] = orders['Cookie Type'].fillna('Unknown')

In [ ]:
def build_dashboard(cookie_types, cities):
    filtered = orders.copy()
    if cookie_types:
        filtered = filtered[filtered['Cookie Type'].isin(cookie_types)]
    if cities:
        filtered = filtered[filtered['City'].isin(cities)]

    total_revenue = filtered['Revenue'].sum()
    total_cost = filtered['Cost'].sum()
    total_profit = filtered['Profit'].sum()
    total_orders = filtered['Order ID'].nunique() if 'Order ID' in filtered.columns else len(filtered)
    avg_order_value = total_revenue / total_orders if total_orders else 0

    kpi = go.Figure()
    kpi.add_trace(go.Indicator(mode='number', title='Revenue', value=total_revenue, number={'prefix': '$'}))
    kpi.add_trace(go.Indicator(mode='number', title='Cost', value=total_cost, number={'prefix': '$'}))
    kpi.add_trace(go.Indicator(mode='number', title='Profit', value=total_profit, number={'prefix': '$'}))
    kpi.add_trace(go.Indicator(mode='number', title='Avg Order Value', value=avg_order_value, number={'prefix': '$'}))
    kpi.update_layout(grid={'rows': 1, 'columns': 4}, template='plotly_white', height=200)

    revenue_by_cookie = filtered.groupby('Cookie Type', as_index=False)['Revenue'].sum().sort_values('Revenue', ascending=False)
    fig_cookie = px.bar(revenue_by_cookie, x='Cookie Type', y='Revenue', title='Revenue by Cookie Type', text='Revenue')
    fig_cookie.update_layout(xaxis_title='Cookie Type', yaxis_title='Revenue', template='plotly_white')

    trends = filtered.groupby('Order Month', as_index=False)[['Revenue', 'Cost', 'Profit']].sum()
    fig_trend = px.line(trends, x='Order Month', y=['Revenue', 'Cost', 'Profit'], title='Monthly Revenue, Cost and Profit')
    fig_trend.update_layout(xaxis_title='Month', yaxis_title='Amount', template='plotly_white')

    city_dist = filtered.groupby('City', as_index=False)['Order ID'].nunique().rename(columns={'Order ID': 'Orders'})
    fig_city = px.pie(city_dist, names='City', values='Orders', title='Order Distribution by City')

    revenue_cost = filtered[['Revenue', 'Cost']].sum().reset_index()
    revenue_cost.columns = ['Metric', 'Amount']
    fig_revenue_cost = px.pie(revenue_cost, names='Metric', values='Amount', title='Revenue vs Cost')

    display(kpi)
    display(fig_cookie)
    display(fig_trend)
    display(fig_city)
    display(fig_revenue_cost)

    top_cookies = revenue_by_cookie.head(10)
    display(top_cookies.style.format({'Revenue': '${:,.0f}'}))

In [ ]:
cookie_options = sorted(orders['Cookie Type'].dropna().unique())
city_options = sorted(orders['City'].dropna().unique())

cookie_selector = widgets.SelectMultiple(
    options=cookie_options,
    value=cookie_options,
    description='Cookie Types',
    rows=6,
    layout=widgets.Layout(width='45%')
)
city_selector = widgets.SelectMultiple(
    options=city_options,
    value=city_options,
    description='Cities',
    rows=6,
    layout=widgets.Layout(width='45%')
)

controls = widgets.HBox([cookie_selector, city_selector])
dashboard_output = widgets.interactive_output(build_dashboard, {'cookie_types': cookie_selector, 'cities': city_selector})

display(widgets.Markdown('### Filters'))
display(controls)
display(dashboard_output)

## Key Insights
- Filter the dashboard by cookie type and city to identify the strongest sales segments.
- Use the time series chart to monitor revenue, cost, and profit trends over time.
- The order distribution pie chart highlights which cities contribute the most orders.
- The revenue vs cost chart reveals margin pressure and profitability balance.

10. Conclusion

The dashboard successfully provides actionable insights into sales performance.
It enables stakeholders to monitor business metrics, identify trends, and make data-driven decisions.
The integration of Python enhances analytical capabilities beyond standard visuals.

11. Dashboard Screenshots

### Overview Dashboard
![Overview](C:/Users/Hp/Desktop/Projects/Task  4/Images/Overview.png)

### Index Page
![Index](C:/Users/Hp/Desktop/Projects/Task  4/Images/Index.png)